# 日频 AlphaMining：Colab A100 一体化训练流水线

本 Notebook 在一次 Colab 会话中依次完成：数据预处理 → 表达式检查 → GFlowNet 训练 → AlphaEval 筛选 → LightGBM 融合 → 训练产物打包下载。

RQAlphaPlus **不在 Colab 运行**。下载本 Notebook 生成的 `alphamining_colab_outputs.zip` 后，在本地使用 `06_rqalpha_backtest.ipynb` 回测。

## 1. 克隆仓库

默认克隆主仓库；目录已存在时自动把本地 `main` 快进到远端最新版，避免 Colab 继续使用旧模块。需要使用其他 fork 时，可预先设置环境变量 `ALPHAMINING_REPO_URL`。

In [1]:
from pathlib import Path
import os

if not Path('configs/training_config.yaml').exists():
    repo = os.environ.get(
        'ALPHAMINING_REPO_URL',
        'https://github.com/JacksonChiy/AlphaMining-GFlowNet-AlphaEval.git',
    )
    !git clone $repo
    %cd AlphaMining-GFlowNet-AlphaEval
else:
    !git checkout main
    !git pull --ff-only origin main

import importlib
importlib.invalidate_caches()
print('当前代码版本:')
!git rev-parse --short HEAD

Cloning into 'AlphaMining-GFlowNet-AlphaEval'...
remote: Enumerating objects: 391, done.
remote: Counting objects: 100% (391/391), done.
remote: Compressing objects: 100% (232/232), done.
remote: Total 391 (delta 225), reused 287 (delta 147), pack-reused 0 (from 0)
Receiving objects: 100% (391/391), 166.54 KiB | 11.89 MiB/s, done.
Resolving deltas: 100% (225/225), done.
/content/AlphaMining-GFlowNet-AlphaEval
当前代码版本:
9567498


## 2. 安装依赖

In [2]:
%pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 136.8 MB/s eta 0:00:00


## 3. 检查 A100、CUDA 与显存

正式训练必须通过 A100 校验。

In [3]:
import torch

print('PyTorch 版本:', torch.__version__)
print('CUDA 状态:', torch.cuda.is_available())
print('CUDA 运行时:', torch.version.cuda)
assert torch.cuda.is_available(), '请在 Colab 中启用 NVIDIA A100 GPU 运行时'
gpu = torch.cuda.get_device_properties(0)
print('GPU 型号:', gpu.name)
print('GPU 显存（GB）:', round(gpu.total_memory / 1024**3, 2))
assert 'A100' in gpu.name.upper(), f'正式训练要求 A100；当前检测到 {gpu.name}'

PyTorch 版本: 2.11.0+cu128
CUDA 状态: True
CUDA 运行时: 12.8
GPU 型号: NVIDIA A100-SXM4-40GB
GPU 显存（GB）: 39.49


## 4. 加载配置

`FAST_MODE=True` 默认读取快速训练配置：使用 2020–2023 年挖掘、筛选和建立初始融合模型，在包含历史预热期的 2020–2026 数据上计算因子，并只输出 2024–2026 年的样本外预测用于本地回测。确认流程可运行后，改为 `False` 即切换到全量正式配置。

In [4]:
import yaml
from src.utils import validate_research_date_split

FAST_MODE = False
REUSE_EXISTING_ALPHA_POOL = False
CONFIG_PATH = ('configs/quick_training_config.yaml' if FAST_MODE
               else 'configs/training_config.yaml')
DOWNLOAD_OUTPUTS = True

with open(CONFIG_PATH, encoding='utf-8') as file:
    config = yaml.safe_load(file)
POOL_SIZE = int(config.get('pipeline', {}).get('pool_size', 100))
print('训练模式:', '快速' if FAST_MODE else '正式全量')
print('配置:', CONFIG_PATH, '因子池:', POOL_SIZE)
print('日期切分:', validate_research_date_split(config))
config

训练模式: 正式全量
配置: configs/training_config.yaml 因子池: 100
日期切分: {'training': '2020-01-01..2023-12-31', 'out_of_sample': '2024-01-01..2026-12-31'}


{'dataset': {'file': 'data/price.csv',
  'output': 'data/daily_price.pkl',
  'horizon': 5,
  'required_columns': ['date',
   'code',
   'open',
   'high',
   'low',
   'close',
   'volume'],
  'start_date': '2020-01-01',
  'end_date': '2026-12-31',
  'mining_start_date': '2020-01-01',
  'mining_end_date': '2023-12-31',
  'out_of_sample_start_date': '2024-01-01',
  'out_of_sample_end_date': '2026-12-31',
  'chunksize': 500000},
 'model': {'name': 'GFlowNet',
  'hidden_dim': 256,
  'num_layers': 4,
  'num_heads': 8,
  'dropout': 0.1,
  'max_sequence_length': 32},
 'training': {'epochs': 100,
  'trajectories_per_epoch': 32,
  'learning_rate': 0.0001,
  'weight_decay': 0.0001,
  'reward_temperature': 1.0,
  'mixed_precision': True,
  'reward_workers': 4,
  'seed': 42},
 'reward': {'top_quantile': 0.1,
  'horizon': 5,
  'risk_aversion': 1.0,
  'min_cross_section': 20,
  'min_coverage': 0.8,
  'coverage_penalty_power': 2.0},
 'alpha_eval': {'rolling_window': 60,
  'perturbation_std': 0.05,
 

## 5. 从 Google Drive 复制 `price.csv`

把行情文件放在 Google Drive 的 `MyDrive/price.csv`。挂载 Drive 后，代码按文件大小判断是否需要复制到 Colab 本地高速磁盘。代码不会调用外部行情接口。

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

PRICE_CSV_DRIVE = Path('/content/drive/MyDrive/price.csv')
PRICE_CSV_LOCAL = Path('data/price.csv')
PRICE_CSV_LOCAL.parent.mkdir(parents=True, exist_ok=True)
if not PRICE_CSV_DRIVE.exists():
    raise FileNotFoundError(f'Google Drive 中没有找到: {PRICE_CSV_DRIVE}')
if (not PRICE_CSV_LOCAL.exists() or
        PRICE_CSV_LOCAL.stat().st_size != PRICE_CSV_DRIVE.stat().st_size):
    print('复制price.csv到Colab本地，请等待...')
    shutil.copy2(PRICE_CSV_DRIVE, PRICE_CSV_LOCAL)
config['dataset']['file'] = str(PRICE_CSV_LOCAL)
print('local csv MB =', round(PRICE_CSV_LOCAL.stat().st_size / 1024**2, 1))

Mounted at /content/drive
复制price.csv到Colab本地，请等待...
local csv MB = 700.7


## 6. 数据预处理

自动完成字段映射、日期转换、排序、缺失与异常处理、可选复权及同日截面标准化，生成 `data/daily_price.pkl`。

In [6]:
import json
from src.data_loader import prepare_price_csv

dataset_filter_keys = (
    'start_date', 'end_date', 'max_stocks', 'universe_start_date',
    'universe_end_date', 'chunksize',
)
dataset_filters = {
    key: config['dataset'][key]
    for key in dataset_filter_keys
    if config['dataset'].get(key) is not None
}
daily = prepare_price_csv(
    config['dataset']['file'],
    config['dataset']['output'],
    'results/data_quality_report.json',
    **dataset_filters,
)
display(daily.head())
print('行数:', len(daily), '交易日数:', daily.date.nunique(), '股票数:', daily.code.nunique())
display(json.loads(Path('results/data_quality_report.json').read_text(encoding='utf-8')))

[Data] filtered_read_start chunksize=500000
[Data] filtered_read_chunk=001 source_rows=500,000 kept_rows=500,000
[Data] filtered_read_chunk=002 source_rows=1,000,000 kept_rows=1,000,000
[Data] filtered_read_chunk=003 source_rows=1,500,000 kept_rows=1,500,000
[Data] filtered_read_chunk=004 source_rows=2,000,000 kept_rows=2,000,000
[Data] filtered_read_chunk=005 source_rows=2,500,000 kept_rows=2,500,000
[Data] filtered_read_chunk=006 source_rows=3,000,000 kept_rows=3,000,000
[Data] filtered_read_chunk=007 source_rows=3,500,000 kept_rows=3,500,000
[Data] filtered_read_chunk=008 source_rows=4,000,000 kept_rows=4,000,000
[Data] filtered_read_chunk=009 source_rows=4,500,000 kept_rows=4,500,000
[Data] filtered_read_chunk=010 source_rows=5,000,000 kept_rows=5,000,000
[Data] filtered_read_chunk=011 source_rows=5,500,000 kept_rows=5,500,000
[Data] filtered_read_chunk=012 source_rows=6,000,000 kept_rows=6,000,000
[Data] filtered_read_chunk=013 source_rows=6,500,000 kept_rows=6,500,000
[Data] filt

,date,code,open,high,low,close,volume,amount,vwap,z_open,z_high,z_low,z_close,z_volume,z_vwap
0,2020-01-02,000001.XSHE,13.0826,13.3183,13.0040,13.2555,153023187.0,2.571196e+09,16.802659,0.086548,0.080455,0.090860,0.087381,3.944691,0.262765
1,2020-01-02,000002.XSHE,26.7856,27.4389,26.5488,26.5896,101213040.0,3.342374e+09,33.023155,1.094384,1.092277,1.100818,1.059321,2.410905,1.358297
2,2020-01-02,000006.XSHE,4.8330,4.8508,4.7796,4.8063,12475176.0,6.745286e+07,5.406967,-0.520198,-0.526290,-0.522386,-0.528492,-0.216087,-0.506900
3,2020-01-02,000007.XSHE,9.5600,9.5800,9.4400,9.5500,3744697.0,3.569469e+07,9.532063,-0.172534,-0.187415,-0.174887,-0.182718,-0.474544,-0.228291
4,2020-01-02,000008.XSHE,3.6480,3.6978,3.6280,3.6978,37746790.0,1.392082e+08,3.687947,-0.607353,-0.608909,-0.608255,-0.609292,0.532053,-0.623003


行数: 7332011 交易日数: 1583 股票数: 5200


{'input_rows': 7332011,
 'source_rows_scanned': 7332011,
 'output_rows': 7332011,
 'column_mapping': {'date': 'date',
  'order_book_id': 'code',
  'open': 'open',
  'high': 'high',
  'low': 'low',
  'close': 'close',
  'volume': 'volume',
  'total_turnover': 'amount'},
 'duplicate_rows_removed': 0,
 'invalid_rows_removed': 0,
 'missing_by_column': {'date': 0,
  'code': 0,
  'open': 0,
  'high': 0,
  'low': 0,
  'close': 0,
  'volume': 0,
  'amount': 0,
  'vwap': 613,
  'z_open': 0,
  'z_high': 0,
  'z_low': 0,
  'z_close': 0,
  'z_volume': 0,
  'z_vwap': 0},
 'outliers_clipped': {'open': 16333,
  'high': 16340,
  'low': 16347,
  'close': 16342,
  'vwap': 17079,
  'volume': 11603,
  'amount': 11603},
 'adjustment_applied': False,
 'risk_fields_available': {'industry': False, 'market_cap': False},
 'date_filter': {'start_date': '2020-01-01', 'end_date': '2026-12-31'},
 'universe_selection': {}}

## 7. 表达式引擎检查

随机生成并执行 10 个表达式，确认复杂度、深度和有效值覆盖率。

In [7]:
import pandas as pd
from src.expression import Expression

expression_rows = []
for seed in range(10):
    expression = Expression.generate(max_depth=4, seed=seed)
    values = expression.execute(daily)
    expression_rows.append({
        'expression': str(expression),
        'complexity': expression.complexity(),
        'depth': expression.depth(),
        'coverage': round(values.notna().mean(), 4),
    })
display(pd.DataFrame(expression_rows))

,expression,complexity,depth,coverage
0,"ts_max(volume,40)",2,2,0.9724
1,open,1,1,1.0000
2,cs_rank(low),2,2,1.0000
3,volume,1,1,1.0000
4,open,1,1,1.0000
5,"ts_zscore(close,10)",2,2,0.9932
6,"cs_zscore(abs(ts_sum(low,5)))",4,4,0.9972
7,log(volume),2,2,1.0000
8,close,1,1,1.0000
9,"sub(vwap,volume)",3,2,0.9999


## 8. GFlowNet A100 混合精度训练

训练使用 Trajectory Balance Loss，开启 A100 混合精度和 TF32，并对同一 epoch 的轨迹执行批量 Transformer 推理；Reward 按配置使用多线程并行，RankIC、LongIR 与风险暴露使用向量化计算。覆盖率同时检查有效观测与有效交易日占比，低覆盖表达式会被降权，低于配置门槛的表达式不会进入因子池。每条训练轨迹仍会输出完整进度，但 GPU 指标按 epoch 批量传回 CPU，减少同步等待。

In [ ]:
from src.gflownet.factor_pool import execute_saved_alpha_pool
from src.gflownet.run_training import run as run_gflownet
from src.utils import create_experiment, slice_date_range

reuse_alpha_pool = (REUSE_EXISTING_ALPHA_POOL and
                    Path('results/alpha_pool.csv').exists())
if reuse_alpha_pool:
    print('检测到已有 Alpha Pool：跳过 GFlowNet 训练，直接在完整行情上重算因子。')
    experiment_dir = create_experiment(CONFIG_PATH)
    execute_saved_alpha_pool(
        daily,
        oos_start_date=config['dataset'].get('out_of_sample_start_date'),
        oos_end_date=config['dataset'].get('out_of_sample_end_date'),
    )
else:
    experiment_dir = run_gflownet(
        CONFIG_PATH, require_a100=True, pool_size=POOL_SIZE
    )
print('实验目录:', experiment_dir.resolve())
if Path('results/gflownet_training_metrics.csv').exists():
    display(pd.read_csv('results/gflownet_training_metrics.csv').tail())
if Path('results/gflownet_trajectory_metrics.csv').exists():
    display(pd.read_csv('results/gflownet_trajectory_metrics.csv').tail(32))
display(pd.read_csv('results/alpha_pool.csv').head(20))

## 9. 检查点重载测试

In [ ]:
from src.gflownet import GFlowNetTrainer, RewardEvaluator

if Path('checkpoints/gflownet_best.pt').exists():
    mining_daily_for_reload = slice_date_range(
        daily, config['dataset'].get('mining_start_date'),
        config['dataset'].get('mining_end_date'), label='检查点样本内行情',
    )
    reward_evaluator = RewardEvaluator(mining_daily_for_reload, **config['reward'])
    loaded_trainer = GFlowNetTrainer.load_checkpoint(
        'checkpoints/gflownet_best.pt', reward_evaluator, device='cuda'
    )
    sample_expression, _, sample_tokens = loaded_trainer.sample_trajectory(greedy=True)
    print('检查点重载成功:', sample_expression)
    print('Token 序列:', sample_tokens)
else:
    print('本次仅复用 Alpha Pool，未提供 GFlowNet checkpoint，跳过重载测试。')
assert Path('results/alpha_pool.csv').exists()
assert Path('results/alpha_factor_matrix.pkl').exists()

## 10. AlphaEval 因子评价与 DPP 筛选

标签统一为 `close(t+5) / close(t+1) - 1`。

In [8]:
!cp /content/drive/MyDrive/results.zip /content/AlphaMining-GFlowNet-AlphaEval
!unzip results.zip

Archive:  results.zip
  inflating: results/alpha_pool.csv  
  inflating: results/alpha_factor_matrix.pkl  
replace results/data_quality_report.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: results/data_quality_report.json  
  inflating: results/gflownet_trajectory_metrics.csv  
  inflating: results/gflownet_training_metrics.csv  
  inflating: results/alpha_factor_matrix_oos.pkl  


In [10]:
import shutil
from src.alpha_eval import AlphaEval, AlphaEvalConfig
from src.utils import slice_date_range

factor_matrix = pd.read_pickle('results/alpha_factor_matrix.pkl')
factor_metadata = pd.read_csv('results/alpha_pool.csv')
alpha_eval_values = dict(config['alpha_eval'])
alpha_eval_values['horizon'] = config['dataset']['horizon']
mining_daily = slice_date_range(
    daily, config['dataset'].get('mining_start_date'),
    config['dataset'].get('mining_end_date'), label='AlphaEval 行情',
)
mining_factor_matrix = slice_date_range(
    factor_matrix, config['dataset'].get('mining_start_date'),
    config['dataset'].get('mining_end_date'), label='AlphaEval 因子',
)
print('AlphaEval 样本内区间:', mining_daily.date.min(), '至', mining_daily.date.max())
alpha_eval = AlphaEval(
    mining_daily, mining_factor_matrix, AlphaEvalConfig(**alpha_eval_values)
)
evaluation = alpha_eval.evaluate(factor_metadata)
selected_factors = evaluation.loc[evaluation['dpp_selected'].astype(bool), 'factor'].tolist()
shutil.copy2('results/alpha_eval_result.csv', experiment_dir / 'alpha_eval_result.csv')
print('DPP 选中因子数:', len(selected_factors))
display(evaluation.head(30))

AlphaEval 样本内区间: 2020-01-02 00:00:00 至 2023-12-29 00:00:00
[AlphaEval] 开始评价：样本=4,207,027，交易日=970，因子=100
[AlphaEval] 因子 1/100 factor_001：开始
[AlphaEval] 因子 1/100 factor_001 [1/4] IC/RankIC 完成 (1.7s)
[AlphaEval] 因子 1/100 factor_001 [2/4] Top组合 Sharpe 完成 (0.4s)
[AlphaEval] 因子 1/100 factor_001 [3/4] 扰动鲁棒性完成 (1.9s)
[AlphaEval] 因子 1/100 factor_001 [4/4] 排名稳定性完成 (2.7s)
[AlphaEval] 因子 1/100 factor_001：完成，耗时=7.1s，总进度=1.0%，预计剩余=11.7min
[AlphaEval] 因子 2/100 factor_002：开始
[AlphaEval] 因子 2/100 factor_002 [1/4] IC/RankIC 完成 (3.0s)
[AlphaEval] 因子 2/100 factor_002 [2/4] Top组合 Sharpe 完成 (0.4s)
[AlphaEval] 因子 2/100 factor_002 [3/4] 扰动鲁棒性完成 (3.0s)
[AlphaEval] 因子 2/100 factor_002 [4/4] 排名稳定性完成 (2.8s)
[AlphaEval] 因子 2/100 factor_002：完成，耗时=9.5s，总进度=2.0%，预计剩余=13.5min
[AlphaEval] 因子 3/100 factor_003：开始
[AlphaEval] 因子 3/100 factor_003 [1/4] IC/RankIC 完成 (2.9s)
[AlphaEval] 因子 3/100 factor_003 [2/4] Top组合 Sharpe 完成 (0.4s)
[AlphaEval] 因子 3/100 factor_003 [3/4] 扰动鲁棒性完成 (3.0s)
[AlphaEval] 因子 3/100 factor_003 [4/4] 排

NameError: name 'experiment_dir' is not defined

## 11. LightGBM 滚动融合

使用 purge 滚动训练。快速配置利用 2020–2023 年的因子和标签建立初始模型，只输出 2024–2026 年的 `prediction_score.csv` 供本地 RQAlphaPlus 回测；后续窗口只使用当时已经成熟的历史标签进行 walk-forward 更新。

In [ ]:
from src.model import LightGBMConfig, LightGBMFusion

fusion = LightGBMFusion(LightGBMConfig(**config['lightgbm']))
prediction = fusion.fit_predict(
    daily, factor_matrix, selected_factors, output_dir='results/lightgbm'
)
shutil.copy2('results/lightgbm/model_metrics.csv', experiment_dir / 'lgbm_model_metrics.csv')
prediction.to_csv(experiment_dir / 'prediction_score.csv', index=False)
loaded_lgbm = LightGBMFusion.load('results/lightgbm/lgbm_model.joblib')
print('LightGBM 模型重载成功；特征数:', len(loaded_lgbm['features']))
display(prediction.tail(30))

## 12. 打包并下载 Colab 训练产物

压缩包包含配置、GFlowNet 检查点、因子池、因子矩阵、AlphaEval 结果、LightGBM 模型与预测分数。下载后在本地仓库根目录解压，再运行 `notebooks/06_rqalpha_backtest.ipynb`。

In [ ]:
from google.colab import files
from zipfile import ZIP_DEFLATED, ZipFile

archive_entries = [
    (Path(CONFIG_PATH), 'results/colab_training_config.yaml'),
    (Path('checkpoints/gflownet_best.pt'), 'checkpoints/gflownet_best.pt'),
    (Path('results/data_quality_report.json'), 'results/data_quality_report.json'),
    (Path('results/gflownet_training_metrics.csv'), 'results/gflownet_training_metrics.csv'),
    (Path('results/gflownet_trajectory_metrics.csv'), 'results/gflownet_trajectory_metrics.csv'),
    (Path('results/alpha_pool.csv'), 'results/alpha_pool.csv'),
    (Path('results/alpha_factor_matrix.pkl'), 'results/alpha_factor_matrix.pkl'),
    (Path('results/alpha_factor_matrix_oos.pkl'), 'results/alpha_factor_matrix_oos.pkl'),
    (Path('results/alpha_eval_result.csv'), 'results/alpha_eval_result.csv'),
    (Path('results/lightgbm/lgbm_model.joblib'), 'results/lightgbm/lgbm_model.joblib'),
    (Path('results/lightgbm/model_metrics.csv'), 'results/lightgbm/model_metrics.csv'),
    (Path('results/lightgbm/feature_importance.csv'), 'results/lightgbm/feature_importance.csv'),
    (Path('results/lightgbm/prediction_score.csv'), 'results/lightgbm/prediction_score.csv'),
]
missing_outputs = [str(path) for path, _ in archive_entries if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(f'缺少训练产物: {missing_outputs}')

archive_path = Path('/content/alphamining_colab_outputs.zip')
with ZipFile(archive_path, 'w', ZIP_DEFLATED) as archive:
    for path, archive_name in archive_entries:
        archive.write(path, arcname=archive_name)
print('产物压缩包:', archive_path, '大小（MB）:', round(archive_path.stat().st_size / 1024**2, 2))
print('本地回测入口: notebooks/06_rqalpha_backtest.ipynb')
if DOWNLOAD_OUTPUTS:
    files.download(str(archive_path))